# Lesson 6: Evaluating Models (Classification Metrics)

How do we *objectively* measure if a classifier is good? This lesson answers the question you asked earlier: **"how do we know which model is right?"**

We'll cover:
1. The **confusion matrix** (the foundation of all metrics).
2. **Accuracy** — and why it can lie.
3. **Precision** and **Recall** — the two that matter most.
4. **F1 score** — balancing them.
5. How moving the **threshold** changes everything.

> Upload to [Google Colab](https://colab.research.google.com) and run top to bottom.

## Step 1: The Confusion Matrix — 4 possible outcomes

For yes/no predictions, every prediction falls into one of 4 boxes:

|  | Predicted POSITIVE | Predicted NEGATIVE |
|---|---|---|
| **Actually POSITIVE** | True Positive (TP) ✓ | False Negative (FN) ✗ missed it |
| **Actually NEGATIVE** | False Positive (FP) ✗ false alarm | True Negative (TN) ✓ |

- **TP**: said pass, was pass. **TN**: said fail, was fail. (both correct)
- **FP**: said pass, was fail. (false alarm / 'Type I error')
- **FN**: said fail, was pass. (missed it / 'Type II error')

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

# Train the same pass/fail model from Lesson 5
X = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]).reshape(-1, 1)
y = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1])

clf = LogisticRegression()
clf.fit(X, y)

y_pred = clf.predict(X)   # uses default 0.5 threshold

cm = confusion_matrix(y, y_pred)
print('Confusion matrix:')
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')

## Step 2: Accuracy — and why it can lie

$$ \text{Accuracy} = \frac{TP + TN}{\text{everything}} = \frac{\text{correct predictions}}{\text{total predictions}} $$

Simple, but **dangerous on imbalanced data**. If 99% of emails are NOT spam, a lazy model that says 'never spam' is 99% accurate — yet useless (it never catches spam!). That's why we need precision and recall.

In [ ]:
print(f'Accuracy: {accuracy_score(y, y_pred):.2f}')

# Demo of the lie: a dataset that is 95% negatives
y_imbalanced = np.array([0]*95 + [1]*5)
y_lazy = np.array([0]*100)   # model that ALWAYS predicts 0
print(f'Lazy model accuracy on imbalanced data: {accuracy_score(y_imbalanced, y_lazy):.2f}  <- looks great, but catches ZERO positives!')

## Step 3: Precision & Recall

$$ \text{Precision} = \frac{TP}{TP + FP} \quad\text{(of those we FLAGGED, how many were right?)} $$

$$ \text{Recall} = \frac{TP}{TP + FN} \quad\text{(of all the ACTUAL positives, how many did we catch?)} $$

**Memory hook:**
- **Precision** = trustworthiness of a positive prediction. High precision = few false alarms.
- **Recall** = coverage of real positives. High recall = few misses.

**The trade-off:** raising the threshold usually raises precision but lowers recall (and vice versa). You can't always max both.

In [ ]:
print(f'Precision: {precision_score(y, y_pred):.2f}')
print(f'Recall:    {recall_score(y, y_pred):.2f}')
print(f'F1 score:  {f1_score(y, y_pred):.2f}')
print()
print(classification_report(y, y_pred, target_names=['FAIL', 'PASS']))

## Step 4: F1 Score — one number to balance both

$$ F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} $$

It's the *harmonic mean* — it stays low unless **both** precision and recall are decent. Great single-number summary when classes are imbalanced.

## Step 5: Watch the threshold change precision vs recall

This connects back to Lesson 5. Lowering the threshold flags more positives -> catches more (recall up) but more false alarms (precision down).

In [ ]:
probs = clf.predict_proba(X)[:, 1]

print(f"{'threshold':>10} {'precision':>10} {'recall':>10} {'f1':>8}")
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    preds = (probs >= t).astype(int)
    p = precision_score(y, preds, zero_division=0)
    r = recall_score(y, preds, zero_division=0)
    f = f1_score(y, preds, zero_division=0)
    print(f'{t:>10.1f} {p:>10.2f} {r:>10.2f} {f:>8.2f}')

## Your turn

1. In the threshold table, find where recall = 1.0 (catches every passer). What happened to precision there?
2. For a **cancer screening** test, would you want high precision or high recall? Why?
3. For a **spam filter** (don't lose important emails), high precision or high recall? Why?
4. Explain in your own words why accuracy is misleading on the 95%-negative dataset.